# Time Series Forecasting: From ARIMA to Prophet

This notebook walks through three approaches to time series forecasting using real housing and inflation data:

1. **ARIMA** (statsmodels) — classical parametric approach
2. **Prophet** (Meta) — decomposition-based, handles seasonality and holidays

**Data:**
- Zillow Home Value Index (ZHVI) — national median home values, monthly
- CPI (CPIAUCSL) from FRED — used as a covariate for context


## 0. Installs and Package Imports



In [ ]:
#Prophet  from Facebook is a popular time series forecasting library 
# try to import packages and install if not found
try:
    import prophet
except ImportError:
    !pip install prophet --quiet
    import prophet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

## 1. Load Data

### 1a. Zillow ZORI Rate (Rental Market)

In [ ]:
# Zillow publishes ZORI as public CSVs. We pull the metro-level file
# and filter for the United States national row.
ZORI_URL = (
    'https://files.zillowstatic.com/research/public_csvs/zori/'
    'Metro_zori_uc_sfrcondomfr_sm_month.csv'
)

zori_raw = pd.read_csv(ZORI_URL)
national = zori_raw[zori_raw['RegionName'] == 'United States'].copy()

# Date columns start after the metadata columns
meta_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
date_cols = [c for c in national.columns if c not in meta_cols]

zori = (
    national[date_cols]
    .T
    .rename(columns={national.index[0]: 'zori'})
)
zori.index = pd.to_datetime(zori.index)
zori.index.name = 'date'
zori = zori.dropna()

print(f'ZORI: {zori.index.min().date()} to {zori.index.max().date()}, {len(zori)} months')
zori.tail(3)

In [ ]:
print(zori.index.min(), zori.index.max())

### 1b. CPI from FRED

In [ ]:
# Pull CPI directly from FRED's public CSV endpoint (no API key required)
CPI_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL'

cpi = pd.read_csv(CPI_URL, parse_dates=['observation_date'], index_col='observation_date')
cpi.index.name = 'date'
cpi.columns = ['cpi']

cpi


In [ ]:
cpi['inflation_yoy'] = cpi['cpi'].pct_change(12, fill_method=None) * 100
cpi

In [ ]:
print(zori.index[:3])
print(cpi.index[:3])

In [ ]:
zori.index = zori.index.to_period('M').to_timestamp('M')
cpi.index = cpi.index.to_period('M').to_timestamp('M')

### 1c. Merge and Trim

In [ ]:
# Align on overlapping monthly dates — ZORI starts ~2015, CPI goes back to 1947
rent_cpi = zori.join(cpi, how='inner').sort_index()

rent_cpi


## 2. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(rent_cpi.index, rent_cpi['zori'], color='steelblue')
axes[0].set_ylabel('Median Rent ($)')
axes[0].set_title('Zillow Observed Rent Index — National Median')

axes[1].plot(rent_cpi.index, rent_cpi['cpi'], color='darkorange')
axes[1].set_ylabel('CPI (Index)')
axes[1].set_title('Consumer Price Index (CPIAUCSL)')

axes[2].plot(rent_cpi.index, rent_cpi['inflation_yoy'], color='firebrick')
axes[2].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[2].axhline(2, color='gray', linewidth=0.8, linestyle=':', label='Fed 2% target')
axes[2].set_ylabel('YoY Inflation (%)')
axes[2].set_title('Year-over-Year Inflation Rate')
axes[2].legend()

fig.suptitle('Rent and Inflation, 2015–Present', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between zori and CPI
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(rent_cpi['cpi'], rent_cpi['zori'] / 1e3, alpha=0.5, s=20, color='steelblue')
ax.set_xlabel('CPI')
ax.set_ylabel('zori ($000s)')
ax.set_title(f'CPI vs. ZORI (r = {rent_cpi["cpi"].corr(rent_cpi["zori"]):.2f})')
plt.tight_layout()
plt.show()

# Discussion: both are trending upward — this is a levels correlation, not necessarily causal.
# In a regression context you'd want to use first differences or log-differences.

### Train / Test Split

We'll hold out the last **24 months** as the test set and forecast into it with each model.

In [ ]:
FORECAST_HORIZON = 24  # months

train = rent_cpi.iloc[:-FORECAST_HORIZON]
test  = rent_cpi.iloc[-FORECAST_HORIZON:]

print(f'Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} months)')
print(f'Test:  {test.index.min().date()}  to {test.index.max().date()}  ({len(test)} months)')

## 3. Approach 1 — ARIMA (statsmodels)

ARIMA is the workhorse of classical time series. It models a series as a function of its own lagged values (AR) and lagged forecast errors (MA), with differencing (I) to handle non-stationarity.

**Key assumptions:**
- Stationarity (after differencing)
- Linearity
- No external covariates in pure ARIMA (you'd need ARIMAX for that)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# Check stationarity on the raw ZORI series
adf_result = adfuller(train['zori'].dropna())
print(f'ADF test on ZORI levels:')
print(f'  Statistic: {adf_result[0]:.4f}')
print(f'  p-value:   {adf_result[1]:.4f}')
print(f'  Conclusion: {"Non-stationary (unit root present)" if adf_result[1] > 0.05 else "Stationary"}')

# Try first differences
adf_diff = adfuller(train['zori'].diff().dropna())
print(f'\nADF test on first differences:')
print(f'  Statistic: {adf_diff[0]:.4f}')
print(f'  p-value:   {adf_diff[1]:.4f}')
print(f'  Conclusion: {"Non-stationary" if adf_diff[1] > 0.05 else "Stationary — d=1 is appropriate"}')

In [ ]:
# Fit ARIMA(2,1,2) — reasonable starting point for monthly housing price data
# Order (p, d, q): AR lags=2, differencing=1, MA lags=2
arima_model = SARIMAX(
    train['zori'],
    order=(2, 1, 2),
    trend='c'
)
arima_fit = arima_model.fit(disp=False)
print(arima_fit.summary().tables[0])

In [ ]:
# Forecast
arima_forecast = arima_fit.get_forecast(steps=FORECAST_HORIZON)
arima_mean = arima_forecast.predicted_mean
arima_ci   = arima_forecast.conf_int(alpha=0.20)  # 80% CI

# Align index to test dates
arima_mean.index = test.index
arima_ci.index   = test.index

# Plot
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index, train['zori'] / 1e3, label='Train', color='steelblue')
ax.plot(test.index,  test['zori'] / 1e3,  label='Actual (test)', color='black', linewidth=1.5)
ax.plot(test.index,  arima_mean / 1e3,    label='ARIMA forecast', color='tomato', linestyle='--')
ax.fill_between(
    test.index,
    arima_ci.iloc[:, 0] / 1e3,
    arima_ci.iloc[:, 1] / 1e3,
    alpha=0.2, color='tomato', label='80% CI'
)
ax.set_ylabel('zori ($000s)')
ax.set_title('ARIMA(2,1,2) Forecast — National zori')
ax.legend()
plt.tight_layout()
plt.show()

# RMSE
arima_rmse = np.sqrt(np.mean((arima_mean.values - test['zori'].values) ** 2))
print(f'ARIMA RMSE: ${arima_rmse:,.0f}')

## 4. Approach 2 — Prophet (Meta)

Prophet decomposes a time series into **trend + seasonality + holidays**. It's more flexible than ARIMA for data with strong seasonal patterns, and it handles missing data and outliers gracefully.

It's also easy to include **regressors** (external variables). We'll add CPI as a regressor here — one of Prophet's practical advantages.

Prophet requires a dataframe with columns `ds` (datestamp) and `y` (target).

In [ ]:
try:
    from prophet import Prophet
except ImportError:
    !pip install prophet --quiet
    from prophet import Prophet

In [ ]:
print(prophet_test[['ds', 'cpi']].isna().sum())
print(prophet_test['ds'].max())
print(rent_cpi['cpi'].index.max())

In [ ]:
future = prophet_test[['ds', 'cpi']].copy()
future['cpi'] = future['cpi'].ffill()
prophet_forecast = prophet_model.predict(future)

In [ ]:

# Prophet format
prophet_train = train.reset_index()[['date', 'zori', 'cpi']].rename(
    columns={'date': 'ds', 'zori': 'y'}
)
prophet_test = test.reset_index()[['date', 'zori', 'cpi']].rename(
    columns={'date': 'ds', 'zori': 'y'}
)

# Fit
prophet_model = Prophet(
    interval_width=0.80,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative'  # better for trending data
)
prophet_model.add_regressor('cpi')  # CPI as external regressor
prophet_model.fit(prophet_train)

print('Prophet model fitted.')

In [ ]:
# For forecasting we need the CPI values in the test period (known future)
# We're treating CPI as a known covariate — in production you'd need to forecast CPI first
future = prophet_test[['ds', 'cpi']].copy()
future['cpi'] = future['cpi'].ffill()
prophet_forecast = prophet_model.predict(future)


# Plot
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index, train['zori'] / 1e3, label='Train', color='steelblue')
ax.plot(test.index,  test['zori'] / 1e3,  label='Actual (test)', color='black', linewidth=1.5)
ax.plot(
    pd.to_datetime(prophet_forecast['ds']),
    prophet_forecast['yhat'] / 1e3,
    label='Prophet forecast', color='mediumseagreen', linestyle='--'
)
ax.fill_between(
    pd.to_datetime(prophet_forecast['ds']),
    prophet_forecast['yhat_lower'] / 1e3,
    prophet_forecast['yhat_upper'] / 1e3,
    alpha=0.2, color='mediumseagreen', label='80% CI'
)
ax.set_ylabel('zori ($000s)')
ax.set_title('Prophet Forecast — National zori (with CPI regressor)')
ax.legend()
plt.tight_layout()
plt.show()

prophet_rmse = np.sqrt(np.mean(
    (prophet_forecast['yhat'].values - test['zori'].values) ** 2
))
print(f'Prophet RMSE: ${prophet_rmse:,.0f}')

In [ ]:
# Prophet's decomposition is one of its best teaching features
from prophet.plot import plot_components

full_future = pd.concat([prophet_train[['ds', 'cpi']], prophet_test[['ds', 'cpi']]])
full_future['cpi'] = full_future['cpi'].ffill()
full_forecast = prophet_model.predict(full_future)

fig = prophet_model.plot_components(full_forecast)
fig.suptitle('Prophet Components — Trend and Seasonality', fontsize=12)
plt.tight_layout()
plt.show()